# ML Term Project – Topic 9: Bank account fraud detection that stays fair and stable over time
### Member 1: data, time-based pipeline, baselines and random forest

Dataset: Bank Account Fraud (BAF) suite, **Base** file (Jesus et al., NeurIPS 2022).

## Step 0 – Project folder in Google Drive

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/BAF_ML_Project"
else:  # running on a laptop / for testing
    PROJECT_DIR = os.environ.get("BAF_PROJECT_DIR", os.path.abspath("BAF_ML_Project"))

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
for d in ["src", "scripts", "data", "outputs/tables", "outputs/figures", "paper/tables", "paper/figures"]:
    os.makedirs(d, exist_ok=True)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print("Working in:", PROJECT_DIR)

## Step 1 – Code
These cells write the python files into the Drive folder (`src/` and `scripts/`). The same files are in the
GitHub repo.

In [ ]:
%%writefile src/__init__.py
# BAF fraud detection project

**`config.py`** – seed, months used for each split, 5% FPR target, tuning budget (20 trials), sampling fraction and the age cut-off for the fairness part.

In [ ]:
%%writefile src/config.py
# settings for the project (seed, months, sampling etc.)
from pathlib import Path

SEED = 42

ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"

TARGET = "fraud_bool"
MONTH_COL = "month"

# months 0-5 are for training/tuning, months 6-7 are the test set (same as the BAF paper)
DEV_MONTHS = [0, 1, 2, 3, 4, 5]
TEST_MONTHS = [6, 7]

# final model is trained on months 0-4 and the 5% FPR threshold is picked on month 5
FINAL_FIT_MONTHS = [0, 1, 2, 3, 4]
THRESHOLD_MONTH = 5

FPR_TARGET = 0.05

# number of random search trials for each model
N_TRIALS = 20

# for training use all fraud rows but only 25% of the non-fraud rows (much faster)
# validation and test months are not sampled
NEG_SAMPLE_FRAC = 0.25

# fairness groups: age over 50 vs 50 and under
AGE_COL = "customer_age"
AGE_CUTOFF = 50

**`data.py`** – loading and cleaning. The datasheet says `-1` means missing in five columns and any negative value means missing in `intended_balcon_amount`. `month` is only used for splitting and `device_fraud_count` is constant, so neither is a feature. Also defines the **six feature groups** (every feature is in exactly one group, which the ablation study needs).

In [ ]:
%%writefile src/data.py
# loading and cleaning the data + the feature groups
import os

import numpy as np
import pandas as pd

from .config import DATA_DIR, TARGET, MONTH_COL

# in these columns -1 means missing (datasheet)
NEG1_MISSING = [
    "prev_address_months_count",
    "current_address_months_count",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w",
]
# here any negative value means missing
NEGATIVE_MISSING = ["intended_balcon_amount"]

CATEGORICAL = ["payment_type", "employment_status", "housing_status", "source", "device_os"]

# not used as features
DROP_ALWAYS = [MONTH_COL, "device_fraud_count"]  # month is only for the split, device_fraud_count is always 0

# each feature is in exactly one group (for the ablation)
FEATURE_GROUPS = {
    "applicant": ["income", "customer_age", "employment_status", "housing_status"],
    "address": ["prev_address_months_count", "current_address_months_count"],
    "email_phone": ["name_email_similarity", "email_is_free", "phone_home_valid", "phone_mobile_valid"],
    "velocity": ["velocity_6h", "velocity_24h", "velocity_4w", "zip_count_4w",
                 "bank_branch_count_8w", "date_of_birth_distinct_emails_4w", "days_since_request"],
    "credit_banking": ["credit_risk_score", "proposed_credit_limit", "intended_balcon_amount",
                       "payment_type", "bank_months_count", "has_other_cards"],
    "device_session": ["device_os", "source", "session_length_in_minutes", "keep_alive_session",
                       "device_distinct_emails_8w", "foreign_request"],
}
ALL_FEATURES = [f for g in FEATURE_GROUPS.values() for f in g]


def load_raw(path=None):
    # reads Base.csv or Base.parquet
    if path is None and os.environ.get("BAF_DATA"):
        path = os.environ["BAF_DATA"]   # path set in the colab notebook
    if path is None:
        pq = DATA_DIR / "Base.parquet"
        path = pq if pq.exists() else DATA_DIR / "Base.csv"
    path = str(path)
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    return pd.read_csv(path)


def clean(df):
    # change the missing value codes to NaN (nothing is learned here so it is ok before splitting)
    df = df.copy()
    for c in NEG1_MISSING:
        df[c] = df[c].where(df[c] != -1)      # becomes float with NaN
    for c in NEGATIVE_MISSING:
        df[c] = df[c].where(df[c] >= 0)
    for c in CATEGORICAL:
        df[c] = df[c].astype(str)
    return df


def load_clean(path=None):
    df = clean(load_raw(path))
    # check that every column is in a group
    leftover = set(df.columns) - set(ALL_FEATURES) - {TARGET} - set(DROP_ALWAYS)
    assert not leftover, f"columns not in any feature group: {leftover}"
    assert len(ALL_FEATURES) == len(set(ALL_FEATURES)), "a feature is in two groups"
    return df


def numeric_features(features):
    return [f for f in features if f not in CATEGORICAL]


def categorical_features(features):
    return [f for f in features if f in CATEGORICAL]


def features_without(groups_to_drop=(), extra_drop=()):
    # feature list without some groups (for the ablation)
    out = []
    for g, cols in FEATURE_GROUPS.items():
        if g in groups_to_drop:
            continue
        out += [c for c in cols if c not in extra_drop]
    return out

**`folds.py`** – the time-based split. Months 6–7 are kept aside. Five forward-chaining folds on months 0–5: train on months 0..k-1, validate on month k. The final run fits on months 0–4 and uses month 5 to fix the threshold.

In [ ]:
%%writefile src/folds.py
# time based splits (months are never shuffled)
# fold 1: train on month 0 -> validate on month 1
# fold 2: train on months 0-1 -> validate on month 2
# ...
# fold 5: train on months 0-4 -> validate on month 5
# months 6 and 7 are only used at the end for testing
import numpy as np

from .config import DEV_MONTHS, TEST_MONTHS, FINAL_FIT_MONTHS, THRESHOLD_MONTH, MONTH_COL


def forward_chaining_folds(df):
    # gives (fold number, train rows, validation rows)
    months = df[MONTH_COL].to_numpy()
    for k, val_m in enumerate(DEV_MONTHS[1:], start=1):
        train_idx = np.where(months < val_m)[0]
        val_idx = np.where(months == val_m)[0]
        yield k, train_idx, val_idx


def final_split(df):
    # rows for the final run: training, threshold month and test months
    months = df[MONTH_COL].to_numpy()
    out = {
        "fit": np.where(np.isin(months, FINAL_FIT_MONTHS))[0],
        "threshold": np.where(months == THRESHOLD_MONTH)[0],
    }
    for m in TEST_MONTHS:
        out[f"test_m{m}"] = np.where(months == m)[0]
    out["test_m6_7"] = np.where(np.isin(months, TEST_MONTHS))[0]
    return out


def describe_folds(df, target="fraud_bool"):
    # size of each fold (for the paper)
    rows = []
    y = df[target].to_numpy()
    for k, tr, va in forward_chaining_folds(df):
        rows.append({
            "fold": k,
            "train_months": f"0-{k-1}" if k > 1 else "0",
            "val_month": k,
            "n_train": len(tr), "train_fraud_rate": y[tr].mean(),
            "n_val": len(va), "val_fraud_rate": y[va].mean(),
        })
    return rows

**`preprocess.py`** – median imputation + a 0/1 *was-missing* column, one-hot encoding for the five categorical columns, scaling for logistic regression / MLP. It is a sklearn pipeline, so it is fitted on the training rows of each fold only. Also the negative sampling of **training rows only** (all fraud + 25% of non-fraud).

In [ ]:
%%writefile src/preprocess.py
# preprocessing (sklearn pipeline, so it is only fitted on the training rows of each fold)
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from .data import numeric_features, categorical_features


def build_preprocessor(features, scale=True, missing_indicators=True):
    # numbers: fill missing values with the median + add a 0/1 missing column, scale if needed
    # categories: one-hot encoding
    # scaling is only needed for logistic regression and the MLP
    num = numeric_features(features)
    cat = categorical_features(features)

    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=missing_indicators))]
    if scale:
        num_steps.append(("scale", StandardScaler()))

    transformers = [("num", Pipeline(num_steps), num)]
    if cat:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat))

    return ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)


def downsample_negatives(X, y, frac, seed):
    # keep all fraud rows and a random part of the non-fraud rows (only used on training data)
    if frac >= 1.0:
        return X, y
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    keep_neg = rng.choice(neg, size=int(round(len(neg) * frac)), replace=False)
    keep = np.sort(np.concatenate([pos, keep_neg]))
    return X.iloc[keep], y[keep]

**`metrics.py`** – recall at 5% false positive rate (main metric), the threshold at a given FPR, ROC-AUC, PR-AUC and F1.

In [ ]:
%%writefile src/metrics.py
# metrics
# main metric is recall at 5% false positive rate (same as the BAF paper)
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

from .config import FPR_TARGET


def threshold_at_fpr(y_true, scores, fpr=FPR_TARGET):
    # threshold where only fpr (5%) of the non-fraud rows have a higher score
    y_true = np.asarray(y_true)
    neg_scores = np.asarray(scores)[y_true == 0]
    return float(np.quantile(neg_scores, 1 - fpr, method="higher"))


def confusion_at(y_true, scores, thr):
    y_true = np.asarray(y_true)
    pred = np.asarray(scores) > thr
    tp = int(np.sum(pred & (y_true == 1)))
    fp = int(np.sum(pred & (y_true == 0)))
    fn = int(np.sum(~pred & (y_true == 1)))
    tn = int(np.sum(~pred & (y_true == 0)))
    return tp, fp, fn, tn


def evaluate(y_true, scores, thr=None, fpr=FPR_TARGET):
    # all the metrics
    # if thr is None the threshold is set on the same rows (used for the cv folds)
    # for the test months the threshold from month 5 is passed in
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if thr is None:
        thr = threshold_at_fpr(y_true, scores, fpr)
    tp, fp, fn, tn = confusion_at(y_true, scores, thr)
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "recall_at_5fpr": recall,
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "precision": precision,
        "f1": f1,
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "accuracy": (tp + tn) / len(y_true),
        "threshold": thr,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

**`models.py`** – the models and their random search spaces.

In [ ]:
%%writefile src/models.py
# models and their random search spaces
from scipy.stats import loguniform, randint, uniform
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


# ---------------- baseline 1: majority class ----------------
def make_majority(params, seed):
    # always predicts "not fraud" -> every row gets the same score
    return DummyClassifier(strategy="most_frequent")


# ---------------- baseline 2: logistic regression scorecard ----------------
def make_logreg(params, seed):
    return LogisticRegression(
        C=params["C"],
        class_weight="balanced",   # fraud class gets more weight
        max_iter=2000,
        solver="lbfgs",
        random_state=seed,
    )


LOGREG_SPACE = {
    "C": loguniform(1e-3, 1e2),
}


# ---------------- random forest ----------------
def make_rf(params, seed):
    return RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        max_samples=params["max_samples"],
        bootstrap=True,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=seed,
    )


RF_SPACE = {
    "n_estimators": randint(100, 301),
    "max_depth": [8, 12, 16, 20, 24],
    "min_samples_leaf": [1, 5, 10, 25, 50, 100],
    "max_features": ["sqrt", 0.2, 0.33],
    "max_samples": uniform(0.05, 0.25),  # each tree uses 5-30% of the rows (otherwise too slow on colab)
}


# name: (function, search space, needs scaling)
MODELS = {
    "majority": (make_majority, None, False),
    "logreg": (make_logreg, LOGREG_SPACE, True),
    "rf": (make_rf, RF_SPACE, False),
}

**`experiment.py`** – cross-validation over the 5 folds, random search, and the final test run.

In [ ]:
%%writefile src/experiment.py
# cross validation on the 5 folds, random search and the final test
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline

from .config import SEED, TARGET, NEG_SAMPLE_FRAC, N_TRIALS
from .data import ALL_FEATURES
from .folds import forward_chaining_folds, final_split
from .metrics import evaluate, threshold_at_fpr
from .models import MODELS
from .preprocess import build_preprocessor, downsample_negatives


def make_pipeline(model_name, params, features, missing_indicators=True, seed=SEED):
    factory, _, needs_scaling = MODELS[model_name]
    pre = build_preprocessor(features, scale=needs_scaling, missing_indicators=missing_indicators)
    return Pipeline([("pre", pre), ("model", factory(params, seed))])


def fit_predict(model_name, params, df, train_idx, pred_idx, features,
                neg_frac=NEG_SAMPLE_FRAC, missing_indicators=True, seed=SEED):
    X_tr = df.iloc[train_idx][features]
    y_tr = df.iloc[train_idx][TARGET].to_numpy()
    X_tr, y_tr = downsample_negatives(X_tr, y_tr, neg_frac, seed)
    pipe = make_pipeline(model_name, params, features, missing_indicators, seed)
    pipe.fit(X_tr, y_tr)
    scores = pipe.predict_proba(df.iloc[pred_idx][features])[:, 1]
    return pipe, scores


def run_cv(model_name, params, df, features=ALL_FEATURES, **kw):
    # one row of metrics for each fold
    rows = []
    for k, tr, va in forward_chaining_folds(df):
        t0 = time.time()
        _, scores = fit_predict(model_name, params, df, tr, va, features, **kw)
        m = evaluate(df.iloc[va][TARGET].to_numpy(), scores)
        m.update({"fold": k, "seconds": round(time.time() - t0, 1)})
        rows.append(m)
    return pd.DataFrame(rows)


def random_search(model_name, df, n_trials=N_TRIALS, seed=SEED, log=print):
    # random search, best trial = highest mean recall at 5% FPR over the 5 folds
    _, space, _ = MODELS[model_name]
    if space is None:  # nothing to tune (majority class)
        candidates = [{}]
    else:
        candidates = list(ParameterSampler(space, n_iter=n_trials, random_state=seed))

    trial_rows, fold_tables = [], []
    for i, params in enumerate(candidates, start=1):
        folds = run_cv(model_name, params, df)
        folds["trial"] = i
        fold_tables.append(folds)
        summary = {"trial": i, **{f"param_{k}": v for k, v in params.items()}}
        for col in ["recall_at_5fpr", "roc_auc", "pr_auc", "f1"]:
            summary[f"{col}_mean"] = folds[col].mean()
            summary[f"{col}_std"] = folds[col].std(ddof=1)
        summary["seconds"] = folds["seconds"].sum()
        trial_rows.append(summary)
        log(f"[{model_name}] trial {i}/{len(candidates)} {params} -> "
            f"recall@5%FPR {summary['recall_at_5fpr_mean']:.4f} ± {summary['recall_at_5fpr_std']:.4f} "
            f"({summary['seconds']:.0f}s)")

    trials = pd.DataFrame(trial_rows)
    best_trial = int(trials.sort_values("recall_at_5fpr_mean", ascending=False).iloc[0]["trial"])
    best_params = candidates[best_trial - 1]
    all_folds = pd.concat(fold_tables, ignore_index=True)
    best_folds = all_folds[all_folds["trial"] == best_trial].copy()
    return trials, best_params, best_folds


def final_test(model_name, params, df, features=ALL_FEATURES, **kw):
    # train on months 0-4, pick the threshold on month 5, then test on months 6 and 7
    idx = final_split(df)
    pred_idx = np.concatenate([idx["threshold"], idx["test_m6_7"]])
    pipe, scores = fit_predict(model_name, params, df, idx["fit"], pred_idx, features, **kw)
    n_thr = len(idx["threshold"])
    s_thr, s_test = scores[:n_thr], scores[n_thr:]

    y = df[TARGET].to_numpy()
    thr = threshold_at_fpr(y[idx["threshold"]], s_thr)

    score_lookup = pd.Series(s_test, index=idx["test_m6_7"])
    rows = []
    for name in ["threshold", "test_m6", "test_m7", "test_m6_7"]:
        s = s_thr if name == "threshold" else score_lookup.loc[idx[name]].to_numpy()
        m = evaluate(y[idx[name]], s, thr=thr)
        m["split"] = {"threshold": "month 5 (threshold)", "test_m6": "month 6",
                      "test_m7": "month 7", "test_m6_7": "months 6-7"}[name]
        rows.append(m)

    test_scores = pd.DataFrame({
        "row": idx["test_m6_7"],
        "month": df.iloc[idx["test_m6_7"]]["month"].to_numpy(),
        "y_true": y[idx["test_m6_7"]],
        "score": s_test,
        "flagged": (s_test > thr).astype(int),
    })
    return pd.DataFrame(rows), test_scores, pipe

**`plotstyle.py`** – one style for all paper figures.

In [ ]:
%%writefile src/plotstyle.py
# colors and settings for all the plots
import matplotlib as mpl
import matplotlib.pyplot as plt

BLUE = "#257de0"     # legitimate / series 1
ORANGE = "#f47230"   # fraud / series 2
AQUA = "#00b894"     # series 3
GREY = "#9a9a9a"
TEXT = "#191919"
TEXT2 = "#666666"

SERIES = [BLUE, ORANGE, AQUA, "#f2a600", "#ee7fac", "#008a17", "#6a4ce2", "#ec4647"]


def use_paper_style():
    mpl.rcParams.update({
        "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
        "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
        "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
        "axes.edgecolor": TEXT2, "axes.labelcolor": TEXT, "text.color": TEXT,
        "xtick.color": TEXT2, "ytick.color": TEXT2,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.6,
        "axes.axisbelow": True, "legend.frameon": False,
        "lines.linewidth": 1.6,
    })


def save(fig, path_no_ext):
    fig.savefig(f"{path_no_ext}.pdf")
    fig.savefig(f"{path_no_ext}.png")
    plt.close(fig)

The scripts below are what the cells further down run (they are also in the repo so the README can reproduce everything).

In [ ]:
%%writefile scripts/00_environment.py
# saves the library versions and the machine info
# python scripts/00_environment.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import json
import os
import platform

import matplotlib
import numpy
import pandas
import scipy
import sklearn

from src.config import TAB_DIR

TAB_DIR.mkdir(parents=True, exist_ok=True)
cpu = platform.processor() or "unknown"
try:
    with open("/proc/cpuinfo") as f:
        names = [l.split(":", 1)[1].strip() for l in f if l.startswith("model name")]
    if names:
        cpu = names[0]
except OSError:
    pass
try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024 ** 3
except (ValueError, OSError, AttributeError):
    ram_gb = None

env = {
    "python": platform.python_version(),
    "numpy": numpy.__version__, "pandas": pandas.__version__, "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__,
    "os": platform.platform(), "cpu": cpu, "cpu_count": os.cpu_count(),
    "ram_gb": round(ram_gb, 1) if ram_gb else None,
    "colab": "google.colab" in sys.modules or os.path.exists("/content"),
}
with open(TAB_DIR / "environment.json", "w") as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))

In [ ]:
%%writefile scripts/01_eda.py
# EDA of the BAF Base dataset (tables go to outputs/tables, plots to outputs/figures)
# python scripts/01_eda.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import FIG_DIR, TAB_DIR, TARGET, AGE_COL, AGE_CUTOFF
from src.data import load_raw, clean, NEG1_MISSING, NEGATIVE_MISSING, CATEGORICAL, FEATURE_GROUPS
from src.folds import describe_folds
from src.plotstyle import use_paper_style, save, BLUE, ORANGE, GREY, TEXT2

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
use_paper_style()

raw = load_raw()
print("shape:", raw.shape)

# ---------- 1. basic checks ----------
overview = {
    "rows": int(len(raw)),
    "columns": int(raw.shape[1]),
    "fraud_cases": int(raw[TARGET].sum()),
    "fraud_rate": float(raw[TARGET].mean()),
    "months": sorted(int(m) for m in raw["month"].unique()),
    "duplicate_rows": int(raw.duplicated().sum()),
    "true_nan_cells": int(raw.isna().sum().sum()),
    "device_fraud_count_unique": sorted(int(v) for v in raw["device_fraud_count"].unique()),
    "n_numeric": int(raw.shape[1] - len(CATEGORICAL) - 2),  # minus target and month
    "n_categorical": len(CATEGORICAL),
}
print(json.dumps(overview, indent=2))
with open(TAB_DIR / "eda_overview.json", "w") as f:
    json.dump(overview, f, indent=2)

# column ranges, to compare with the datasheet
num_cols = raw.select_dtypes("number").columns
ranges = pd.DataFrame({"min": raw[num_cols].min(), "max": raw[num_cols].max(),
                       "n_unique": raw[num_cols].nunique()})
ranges.to_csv(TAB_DIR / "eda_column_ranges.csv")

# negative values in columns where they are not missing codes
for c in ["velocity_6h", "credit_risk_score"]:
    print(f"{c}: {(raw[c] < 0).sum()} negative values ({(raw[c] < 0).mean():.4%})")

# ---------- 2. per month ----------
per_month = raw.groupby("month").agg(n=(TARGET, "size"), fraud=(TARGET, "sum"))
per_month["fraud_rate"] = per_month["fraud"] / per_month["n"]
per_month["split"] = ["dev"] * 6 + ["test"] * 2
per_month.to_csv(TAB_DIR / "eda_per_month.csv")
print(per_month)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.3))
colors = [BLUE] * 6 + [ORANGE] * 2
axes[0].bar(per_month.index, per_month["n"] / 1000, color=colors, width=0.7)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Applications (thousands)")
axes[0].set_title("(a) Applications per month")
axes[1].bar(per_month.index, per_month["fraud_rate"] * 100, color=colors, width=0.7)
axes[1].axhline(overview["fraud_rate"] * 100, color=GREY, lw=1, ls="--")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].set_title("(b) Fraud rate per month")
for ax in axes:
    ax.set_xticks(range(8))
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=BLUE, label="Development months (0-5)"),
                    Patch(color=ORANGE, label="Test months (6-7)"),
                    Line2D([0], [0], color=GREY, ls="--", lw=1,
                           label=f"Overall fraud rate ({overview['fraud_rate']*100:.2f}%)")],
           loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.88))
save(fig, FIG_DIR / "eda_months")

# ---------- 3. missing values ----------
df = clean(raw)
miss_cols = NEG1_MISSING + NEGATIVE_MISSING
miss_rows = []
for c in miss_cols:
    is_miss = df[c].isna()
    miss_rows.append({
        "feature": c,
        "missing_rate": is_miss.mean(),
        "fraud_rate_if_missing": df.loc[is_miss, TARGET].mean(),
        "fraud_rate_if_present": df.loc[~is_miss, TARGET].mean(),
    })
miss = pd.DataFrame(miss_rows).sort_values("missing_rate", ascending=False)
miss.to_csv(TAB_DIR / "eda_missing.csv", index=False)
print(miss)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.4), gridspec_kw={"width_ratios": [1, 1.25]})
y = np.arange(len(miss))
short = [c.replace("_count", "").replace("_in_minutes", "").replace("_months", "_mo")
         for c in miss["feature"]]
axes[0].barh(y, miss["missing_rate"] * 100, color=BLUE, height=0.6)
axes[0].set_yticks(y, short)
axes[0].invert_yaxis()
axes[0].set_xlabel("Rows with value missing (%)")
axes[0].set_title("(a) Share of missing values")
for yi, v in zip(y, miss["missing_rate"] * 100):
    axes[0].text(v + 1, yi, f"{v:.1f}%", va="center", fontsize=6.5, color=TEXT2)
axes[0].set_xlim(0, 100)
h = 0.36
axes[1].barh(y - h / 2, miss["fraud_rate_if_present"] * 100, height=h, color=BLUE, label="Value present")
axes[1].barh(y + h / 2, miss["fraud_rate_if_missing"] * 100, height=h, color=ORANGE, label="Value missing")
axes[1].set_yticks(y, [""] * len(y))
axes[1].invert_yaxis()
axes[1].set_xlabel("Fraud rate (%)")
axes[1].set_title("(b) Fraud rate when missing vs. present")
fig.legend(handles=[Patch(color=BLUE, label="Value present"), Patch(color=ORANGE, label="Value missing")],
           loc="upper center", ncol=2, bbox_to_anchor=(0.72, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.9))
save(fig, FIG_DIR / "eda_missing")

# ---------- 4. age groups (protected attribute) ----------
age = raw.groupby(AGE_COL).agg(n=(TARGET, "size"), fraud=(TARGET, "sum"))
age["fraud_rate"] = age["fraud"] / age["n"]
age["share"] = age["n"] / len(raw)
age.to_csv(TAB_DIR / "eda_age.csv")
older = raw[AGE_COL] > AGE_CUTOFF
age_groups = {
    "over_50_share": float(older.mean()),
    "over_50_fraud_rate": float(raw.loc[older, TARGET].mean()),
    "50_or_under_fraud_rate": float(raw.loc[~older, TARGET].mean()),
}
with open(TAB_DIR / "eda_age_groups.json", "w") as f:
    json.dump(age_groups, f, indent=2)
print(age, age_groups)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.3))
col = [ORANGE if a > AGE_CUTOFF else BLUE for a in age.index]
axes[0].bar(age.index, age["share"] * 100, width=6, color=col)
axes[0].set_xlabel("customer_age (decade)")
axes[0].set_ylabel("Share of applications (%)")
axes[0].set_title("(a) Age distribution")
axes[1].bar(age.index, age["fraud_rate"] * 100, width=6, color=col)
axes[1].set_xlabel("customer_age (decade)")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].set_title("(b) Fraud rate by age")
for ax in axes:
    ax.set_xticks(age.index)
fig.legend(handles=[Patch(color=BLUE, label=f"Age \u2264 {AGE_CUTOFF}"),
                    Patch(color=ORANGE, label=f"Age > {AGE_CUTOFF} (fairness group)")],
           loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.88))
save(fig, FIG_DIR / "eda_age")

# ---------- 5. a few features by class ----------
show = ["name_email_similarity", "credit_risk_score", "current_address_months_count",
        "velocity_6h"]
fig, axes = plt.subplots(1, 4, figsize=(7.0, 1.9))
for ax, c in zip(axes, show):
    vals = df[c].dropna()
    bins = np.linspace(vals.quantile(0.001), vals.quantile(0.999), 40)
    for lab, colr, name in [(0, BLUE, "Legitimate"), (1, ORANGE, "Fraud")]:
        ax.hist(df.loc[df[TARGET] == lab, c].dropna(), bins=bins, density=True,
                histtype="step", lw=1.4, color=colr, label=name)
    ax.set_title(c.replace("_months_count", "_months"), fontsize=7)
    ax.set_yticks([])
    ax.set_xlabel("value")
axes[0].set_ylabel("Density")
fig.legend(handles=[Patch(color=BLUE, label="Legitimate"), Patch(color=ORANGE, label="Fraud")],
           loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.86))
save(fig, FIG_DIR / "eda_features_by_class")

# ---------- 6. drift: how much do feature means move per month ----------
num_feats = [c for g in FEATURE_GROUPS.values() for c in g if c not in CATEGORICAL]
dev = df[df["month"] <= 5]
z = (df.groupby("month")[num_feats].mean() - dev[num_feats].mean()) / dev[num_feats].std()
z.T.to_csv(TAB_DIR / "eda_monthly_shift_z.csv")
top = z.loc[[6, 7]].abs().max().sort_values(ascending=False).head(5).index.tolist()
print("features whose monthly mean moves most (in dev std units):", top)

fig, ax = plt.subplots(figsize=(3.4, 2.9))
for i, c in enumerate(top):
    ax.plot(z.index, z[c], marker="o", ms=3, color=[BLUE, ORANGE, "#00b894", "#f2a600", "#6a4ce2"][i], label=c)
ax.axvspan(5.5, 7.5, color="#f0f0f0", zorder=0)
ax.text(6.5, ax.get_ylim()[1] * 0.92, "test", ha="center", fontsize=6.5, color=TEXT2)
ax.axhline(0, color=GREY, lw=0.8)
ax.set_xticks(range(8))
ax.set_xlabel("Month")
ax.set_ylabel("Mean shift (std. of months 0-5)")
ax.set_title("Monthly drift of the most-shifting features")
ax.legend(fontsize=5.8, loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2)
fig.tight_layout()
save(fig, FIG_DIR / "eda_drift")

# ---------- 7. fold sizes ----------
folds = pd.DataFrame(describe_folds(df))
folds.to_csv(TAB_DIR / "fold_sizes.csv", index=False)
print(folds)
print("EDA done.")

In [ ]:
%%writefile scripts/02_tune_cv.py
# random search for one model on the 5 folds
# python scripts/02_tune_cv.py --model logreg
# saves cv_trials_<model>.csv, cv_folds_<model>.csv and best_params_<model>.json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import argparse
import json
import time

import numpy as np

from src.config import TAB_DIR, SEED, N_TRIALS, NEG_SAMPLE_FRAC
from src.data import load_clean
from src.experiment import random_search
from src.models import MODELS

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True, choices=list(MODELS))
parser.add_argument("--trials", type=int, default=N_TRIALS)
args = parser.parse_args()

np.random.seed(SEED)
TAB_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
df = load_clean()
print(f"loaded {len(df)} rows; negative sampling frac for training = {NEG_SAMPLE_FRAC}")

trials, best_params, best_folds = random_search(args.model, df, n_trials=args.trials)

trials.to_csv(TAB_DIR / f"cv_trials_{args.model}.csv", index=False)
best_folds["model"] = args.model
best_folds.to_csv(TAB_DIR / f"cv_folds_{args.model}.csv", index=False)


def to_py(v):  # numpy types -> plain python for json
    return v.item() if hasattr(v, "item") else v


with open(TAB_DIR / f"best_params_{args.model}.json", "w") as f:
    json.dump({k: to_py(v) for k, v in best_params.items()}, f, indent=2)

print("\nbest params:", best_params)
print(best_folds[["fold", "recall_at_5fpr", "roc_auc", "pr_auc", "f1"]].to_string(index=False))
print(f"total time {(time.time() - t0) / 60:.1f} min")

In [ ]:
%%writefile scripts/03_final_test.py
# final run with the best params: train on months 0-4, threshold on month 5, test on months 6 and 7
# python scripts/03_final_test.py --model logreg
# saves test_<model>.csv and test_scores_<model>.csv
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import argparse
import json

import numpy as np

from src.config import TAB_DIR, SEED
from src.data import load_clean
from src.experiment import final_test
from src.models import MODELS

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True, choices=list(MODELS))
args = parser.parse_args()
np.random.seed(SEED)

with open(TAB_DIR / f"best_params_{args.model}.json") as f:
    params = json.load(f)

df = load_clean()
metrics, test_scores, _ = final_test(args.model, params, df)
metrics["model"] = args.model
metrics.to_csv(TAB_DIR / f"test_{args.model}.csv", index=False)
test_scores.to_csv(TAB_DIR / f"test_scores_{args.model}.csv", index=False)

cols = ["split", "recall_at_5fpr", "fpr", "precision", "f1", "roc_auc", "pr_auc", "threshold"]
print(metrics[cols].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
%%writefile scripts/04_results_table.py
# puts every model that has results into one table (cv mean ± std + test months)
# python scripts/04_results_table.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import pandas as pd

from src.config import TAB_DIR

ORDER = ["majority", "logreg", "rf", "lightgbm", "catboost", "mlp"]
NAMES = {"majority": "Majority class", "logreg": "Logistic regression", "rf": "Random forest",
         "lightgbm": "LightGBM", "catboost": "CatBoost", "mlp": "MLP"}
METRICS = [("recall_at_5fpr", "Recall@5%FPR"), ("roc_auc", "ROC-AUC"),
           ("pr_auc", "PR-AUC"), ("f1", "F1")]

rows = []
for m in ORDER:
    f = TAB_DIR / f"cv_folds_{m}.csv"
    if not f.exists():
        continue
    folds = pd.read_csv(f)
    row = {"model": NAMES[m]}
    for col, label in METRICS:
        row[f"CV {label}"] = f"{folds[col].mean():.3f} ± {folds[col].std(ddof=1):.3f}"
    t = TAB_DIR / f"test_{m}.csv"
    if t.exists():
        test = pd.read_csv(t).set_index("split")
        row["Test M6 Recall@5%FPR"] = f"{test.loc['month 6', 'recall_at_5fpr']:.3f}"
        row["Test M7 Recall@5%FPR"] = f"{test.loc['month 7', 'recall_at_5fpr']:.3f}"
        row["Test M6-7 FPR"] = f"{test.loc['months 6-7', 'fpr']:.3f}"
        row["Test M6-7 ROC-AUC"] = f"{test.loc['months 6-7', 'roc_auc']:.3f}"
    rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(TAB_DIR / "results_table.csv", index=False)
with open(TAB_DIR / "results_table.md", "w") as f:
    f.write(table.to_markdown(index=False))
with open(TAB_DIR / "results_table.tex", "w") as f:
    f.write(table.to_latex(index=False).replace("±", r"$\pm$").replace("%", r"\%"))
print(table.to_markdown(index=False))

In [ ]:
%%writefile scripts/05_paper_tables.py
# makes the latex tables for the paper from the result files
# python scripts/05_paper_tables.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import json

import pandas as pd

from src.config import TAB_DIR, ROOT
from src.data import FEATURE_GROUPS

OUT = ROOT / "paper" / "tables"
OUT.mkdir(parents=True, exist_ok=True)


def esc(s):
    return str(s).replace("_", r"\_").replace("%", r"\%")


# ---------------- fold sizes ----------------
folds = pd.read_csv(TAB_DIR / "fold_sizes.csv")
lines = [r"\begin{table}[!t]", r"\centering",
         r"\caption{Forward-chaining folds on the development months (training sizes before sampling).}",
         r"\label{tab:folds}", r"\footnotesize", r"\setlength{\tabcolsep}{3.5pt}",
         r"\begin{tabular}{ccrrrr}", r"\hline",
         r"Fold & Train mon. & Train rows & Fraud & Val. mon. & Fraud \\", r"\hline"]
for _, r in folds.iterrows():
    tm = "0" if r.fold == 1 else f"0--{r.fold - 1}"
    lines.append(f"{r.fold} & {tm} & {int(r.n_train):,} & {r.train_fraud_rate*100:.2f}\\% & "
                 f"{int(r.val_month)} & {r.val_fraud_rate*100:.2f}\\% \\\\")
lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
(OUT / "fold_sizes.tex").write_text("\n".join(lines) + "\n")

# ---------------- tuning ranges + chosen values ----------------
RANGES = {
    "logreg": [("C", "log-unif. $[10^{-3}, 10^{2}]$", "C")],
    "rf": [("n\\_estimators", "integer $[100, 300]$", "n_estimators"),
           ("max\\_depth", "\\{8, 12, 16, 20, 24\\}", "max_depth"),
           ("min\\_samples\\_leaf", "\\{1, 5, 10, 25, 50, 100\\}", "min_samples_leaf"),
           ("max\\_features", "\\{sqrt, 0.2, 0.33\\}", "max_features"),
           ("max\\_samples", "uniform $[0.05, 0.30]$", "max_samples")],
}
NAMES = {"logreg": "Log. reg.", "rf": "Rand. forest"}


def fmt(v):
    if isinstance(v, float):
        return f"{v:.3g}" if v < 1 else f"{v:.2f}"
    return esc(v)


lines = [r"\begin{table}[!t]", r"\centering",
         r"\caption{Random-search ranges (20 trials per model) and selected values.}",
         r"\label{tab:tuning}", r"\footnotesize", r"\setlength{\tabcolsep}{3.5pt}",
         r"\begin{tabular}{llll}", r"\hline",
         r"Model & Hyperparameter & Search range & Selected \\", r"\hline"]
for m, rows in RANGES.items():
    f = TAB_DIR / f"best_params_{m}.json"
    best = json.loads(f.read_text()) if f.exists() else {}
    for i, (label, rng, key) in enumerate(rows):
        name = NAMES[m] if i == 0 else ""
        lines.append(f"{name} & {label} & {rng} & {fmt(best.get(key, '--'))} \\\\")
    lines.append(r"\hline")
lines += [r"\end{tabular}", r"\end{table}"]
(OUT / "tuning_table.tex").write_text("\n".join(lines) + "\n")

# ---------------- feature groups ----------------
lines = [r"\begin{table}[!t]", r"\centering",
         r"\caption{Feature groups used in the dataset description and the ablation study.}",
         r"\label{tab:groups}", r"\footnotesize", r"\setlength{\tabcolsep}{3.5pt}",
         r"\begin{tabular}{>{\raggedright\arraybackslash}p{1.55cm}>{\raggedright\arraybackslash}p{6.3cm}}", r"\hline",
         r"Group & Features \\", r"\hline"]
pretty = {"applicant": "Applicant", "address": "Address history", "email_phone": "Email and phone",
          "velocity": "Application velocity", "credit_banking": "Credit and banking",
          "device_session": "Device and session"}
for g, cols in FEATURE_GROUPS.items():
    lines.append(f"{pretty[g]} ({len(cols)}) & " + ", ".join(r"\texttt{\footnotesize " + esc(c).replace("\\_", "\\_\\allowbreak ") + "}" for c in cols) + r" \\")
lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
(OUT / "feature_groups.tex").write_text("\n".join(lines) + "\n")

# ---------------- baseline results (CV + test) ----------------
order = [("majority", "Majority class"), ("logreg", "Logistic regression"), ("rf", "Random forest")]
lines = [r"\begin{table*}[!t]", r"\centering",
         r"\caption{Baselines and random forest: mean $\pm$ std over the five validation folds, and the "
         r"test months with the threshold fixed on month 5. (To be merged into the main results table.)}",
         r"\label{tab:member1-results}", r"\footnotesize",
         r"\begin{tabular}{lcccc|ccc}", r"\hline",
         r" & \multicolumn{4}{c|}{Validation folds (months 1--5)} & \multicolumn{3}{c}{Test} \\",
         r"Model & Recall@5\%FPR & ROC-AUC & PR-AUC & F1 & Recall M6 & Recall M7 & FPR M6--7 \\", r"\hline"]
for m, name in order:
    f = TAB_DIR / f"cv_folds_{m}.csv"
    if not f.exists():
        continue
    cv = pd.read_csv(f)
    cells = [f"{cv[c].mean():.3f} $\\pm$ {cv[c].std(ddof=1):.3f}" for c in ["recall_at_5fpr", "roc_auc", "pr_auc", "f1"]]
    t = pd.read_csv(TAB_DIR / f"test_{m}.csv").set_index("split")
    cells += [f"{t.loc['month 6', 'recall_at_5fpr']:.3f}", f"{t.loc['month 7', 'recall_at_5fpr']:.3f}",
              f"{t.loc['months 6-7', 'fpr']*100:.2f}\\%"]
    lines.append(name + " & " + " & ".join(cells) + r" \\")
lines += [r"\hline", r"\end{tabular}", r"\end{table*}"]
(OUT / "member1_results.tex").write_text("\n".join(lines) + "\n")

print("wrote:", ", ".join(p.name for p in sorted(OUT.glob("*.tex"))))

In [ ]:
%%writefile scripts/pipeline_diagram.py
# draws the pipeline diagram for the methodology section
# python scripts/pipeline_diagram.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from src.config import FIG_DIR
from src.plotstyle import use_paper_style, save, BLUE, ORANGE, TEXT, TEXT2

use_paper_style()
FIG_DIR.mkdir(parents=True, exist_ok=True)

W, H = 7.16, 3.05           # IEEE two-column width in inches
fig = plt.figure(figsize=(W, H))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.axis("off")

BW, BH = 1.02, 0.50          # box size
FILL_KEY = "#e8f1fc"
FILL_TEST = "#fdeee6"


def box(x, y, text, fill="white", w=BW, h=BH, edge=TEXT2):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.06",
                                fc=fill, ec=edge, lw=0.7))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=6.3, color=TEXT,
            linespacing=1.15)
    return (x, y, w, h)


def arrow(p, q, style="-|>", ls="-"):
    ax.annotate("", xy=q, xytext=p,
                arrowprops=dict(arrowstyle=style, color=TEXT2, lw=0.7, ls=ls,
                                shrinkA=0, shrinkB=0, mutation_scale=7))


def right(b):  return (b[0] + b[2], b[1] + b[3] / 2)
def left(b):   return (b[0], b[1] + b[3] / 2)
def bottom(b): return (b[0] + b[2] / 2, b[1])
def top(b):    return (b[0] + b[2] / 2, b[1] + b[3])


gap = (W - 0.16 - 6 * BW) / 5          # horizontal gap between 6 boxes
xs = [0.08 + i * (BW + gap) for i in range(6)]

# ---- row 1: data preparation ----
y1 = 2.42
b_data = box(xs[0], y1, "BAF Base dataset\n1M applications\nmonths 0-7", FILL_KEY)
b_clean = box(xs[1], y1, "Cleaning\nmissing codes -> NaN\ndrop 2 non-features")
b_split = box(xs[2], y1, "Time split by month\n0-5: development\n6-7: test (locked)")
arrow(right(b_data), left(b_clean))
arrow(right(b_clean), left(b_split))

# ---- row 2: development loop ----
y2 = 1.30
ax.add_patch(FancyBboxPatch((0.03, y2 - 0.13), W - 0.085, BH + 0.40,
                            boxstyle="round,pad=0.0,rounding_size=0.08",
                            fc="none", ec=BLUE, lw=0.8, ls=(0, (4, 2))))
ax.text(xs[0] + BW / 2 + 0.08, y2 + BH + 0.14, "Development (months 0-5): repeated for every model and every tuning trial",
        fontsize=6.3, color=BLUE, va="center")
b_folds = box(xs[0], y2, "5 forward-chaining\nfolds: train 0..k-1,\nvalidate month k")
b_samp = box(xs[1], y2, "Training rows only:\nall fraud + 25%\nof non-fraud")
b_pre = box(xs[2], y2, "Preprocessing fit on\ntrain rows: impute,\nmissing flags, one-hot")
b_model = box(xs[3], y2, "Model: Majority, LR,\nRF, LightGBM,\nCatBoost, MLP", FILL_KEY)
b_score = box(xs[4], y2, "Score validation\nmonth: Recall@5%FPR,\nROC-AUC, PR-AUC, F1")
b_tune = box(xs[5], y2, "Random search\n20 trials; best mean\nRecall@5%FPR")
for a, b in [(b_folds, b_samp), (b_samp, b_pre), (b_pre, b_model), (b_model, b_score), (b_score, b_tune)]:
    arrow(right(a), left(b))

# split -> folds (elbow)
sx, sy = bottom(b_split)
fx, fy = top(b_folds)
mid = (y1 + y2 + BH) / 2 + 0.02
ax.plot([sx, sx, fx], [sy, mid, mid], color=TEXT2, lw=0.7)
arrow((fx, mid), (fx, fy + 0.0))
ax.text(sx + 0.05, mid + 0.08, "months 0-5", fontsize=5.8, color=TEXT2)

# ---- row 3: final run ----
y3 = 0.12
b_fit = box(xs[2], y3, "Refit best config\non months 0-4")
b_thr = box(xs[3], y3, "Fix threshold at\n5% FPR on month 5")
b_test = box(xs[4], y3, "Evaluate months\n6 and 7 with the\nfixed threshold", FILL_TEST, edge=ORANGE)
b_out = box(xs[5], y3, "Analysis: RQ1-RQ3,\nfairness, ablation,\nerror analysis", FILL_KEY)
arrow(right(b_fit), left(b_thr))
arrow(right(b_thr), left(b_test))
arrow(right(b_test), left(b_out))

# tune -> refit (elbow)
tx, ty = bottom(b_tune)
rx, ry = top(b_fit)
mid2 = (y2 - 0.13 + y3 + BH) / 2
ax.plot([tx, tx, rx], [ty, mid2, mid2], color=TEXT2, lw=0.7)
arrow((rx, mid2), (rx, ry))
ax.text(tx - 0.05, mid2 + 0.07, "best hyperparameters", fontsize=5.8, color=TEXT2, ha="right")

# split -> test (dashed, the test months are only touched at the end)
ex = b_split[0] + b_split[2] + 0.06
ax.plot([right(b_split)[0], ex + gap * 0.0 + 0.30], [right(b_split)[1]] * 2, color=ORANGE, lw=0.7, ls=(0, (3, 2)))
tx2 = b_test[0] + b_test[2] * 0.80
ax.plot([ex + 0.30, W - 0.02, W - 0.02, tx2], [right(b_split)[1], right(b_split)[1], y3 + BH + 0.10, y3 + BH + 0.10],
        color=ORANGE, lw=0.7, ls=(0, (3, 2)))
ax.annotate("", xy=(tx2, y3 + BH), xytext=(tx2, y3 + BH + 0.10),
            arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=0.7, mutation_scale=7, shrinkA=0, shrinkB=0))
ax.text(xs[3] + 0.05, right(b_split)[1] + 0.06, "months 6-7 held out until the final run",
        fontsize=5.8, color=ORANGE)

save(fig, FIG_DIR / "pipeline")
print("saved", FIG_DIR / "pipeline.pdf")

## Step 2 – Load the dataset
`Base.csv` has to be in `MyDrive/BAF_ML_Project/data/` (only Base.csv is needed from the Kaggle zip, not the
Variant files). It is read once and saved as parquet because that loads much faster in the later steps.

In [ ]:
import pandas as pd

csv_path = os.path.join(PROJECT_DIR, "data", "Base.csv")
local_pq = "/content/Base.parquet" if IN_COLAB else os.path.join(PROJECT_DIR, "data", "Base_local.parquet")
if os.environ.get("BAF_TEST_DATA"):          # only used when testing the notebook offline
    csv_path = os.environ["BAF_TEST_DATA"]

assert os.path.exists(csv_path), "Base.csv not found - upload it to MyDrive/BAF_ML_Project/data/ first"
print("Reading", csv_path)
raw = pd.read_csv(csv_path)
raw.to_parquet(local_pq, index=False)
os.environ["BAF_DATA"] = local_pq             # src/data.py reads from here
print(raw.shape)
raw.head()

### Check that this is the right file (numbers from the BAF paper / datasheet)

In [ ]:
print("rows:", len(raw), "| columns:", raw.shape[1])
print("fraud rate: %.4f%%" % (raw["fraud_bool"].mean() * 100))
print("months:", sorted(raw["month"].unique()))
if not os.environ.get("BAF_TEST_DATA"):
    assert raw.shape == (1_000_000, 32), "Expected 1,000,000 rows x 32 columns - is this Base.csv?"
    assert sorted(raw["month"].unique()) == list(range(8))
    assert 0.010 < raw["fraud_bool"].mean() < 0.012
    print("OK - matches the BAF Base dataset")
del raw

## Step 3 – Environment and EDA

In [ ]:
%run scripts/00_environment.py

In [ ]:
%run scripts/01_eda.py

In [ ]:
from IPython.display import Image, Markdown, display
for f in ["eda_months", "eda_age", "eda_missing", "eda_features_by_class", "eda_drift"]:
    display(Image(f"outputs/figures/{f}.png", width=760))

In [ ]:
for t in ["eda_per_month", "eda_missing", "eda_age", "fold_sizes"]:
    print("\n==", t)
    display(pd.read_csv(f"outputs/tables/{t}.csv"))

## Step 4 – Baselines: majority class and logistic regression scorecard
Each model: 20-trial random search on the 5 forward-chaining folds (score = mean recall at 5% FPR), then the final
run (fit months 0–4, threshold on month 5, test months 6 and 7). Models that already have results in Drive are skipped.

In [ ]:
import time

def run_model(name):
    if os.path.exists(f"outputs/tables/test_{name}.csv"):
        print(f"{name}: results already saved - skipping (delete outputs/tables/*_{name}.* to rerun)")
        return
    t0 = time.time()
    get_ipython().run_line_magic("run", f"scripts/02_tune_cv.py --model {name}")
    get_ipython().run_line_magic("run", f"scripts/03_final_test.py --model {name}")
    print(f"{name} finished in {(time.time() - t0) / 60:.1f} min")

run_model("majority")
run_model("logreg")

## Step 5 – Random forest (the slow one, ~45–60 min on the free CPU runtime)

In [ ]:
run_model("rf")

## Step 6 – Results table (mean ± std over folds, and test months)

In [ ]:
%run scripts/04_results_table.py
display(Markdown(open("outputs/tables/results_table.md").read()))

In [ ]:
for m in ["majority", "logreg", "rf"]:
    print("\n==", m, "- test months (threshold fixed on month 5)")
    t = pd.read_csv(f"outputs/tables/test_{m}.csv")
    display(t[["split", "recall_at_5fpr", "fpr", "precision", "f1", "roc_auc", "pr_auc", "threshold"]].round(4))

## Step 7 – Tables and figures for the paper

In [ ]:
%run scripts/05_paper_tables.py
%run scripts/pipeline_diagram.py
import shutil, glob
for f in glob.glob("outputs/figures/*.pdf"):
    shutil.copy(f, "paper/figures/")
display(Image("outputs/figures/pipeline.png", width=760))

## Step 8 – Summary of the numbers used in the write-up

In [ ]:
import json
summary = {
    "overview": json.load(open("outputs/tables/eda_overview.json")),
    "age_groups": json.load(open("outputs/tables/eda_age_groups.json")),
    "environment": json.load(open("outputs/tables/environment.json")),
    "best_params": {m: json.load(open(f"outputs/tables/best_params_{m}.json")) for m in ["majority", "logreg", "rf"]},
}
json.dump(summary, open("outputs/tables/summary_member1.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
shutil.make_archive(os.path.join(PROJECT_DIR, "member1_outputs"), "zip", PROJECT_DIR, "outputs")
print("\nAll outputs saved in", PROJECT_DIR, "(and zipped as member1_outputs.zip)")